In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [2]:
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [4]:
import zipfile
import os
zip_path = "/content/tinyllama-lora-1.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

In [5]:
model_path = "/content/checkpoint-5"

In [6]:
!pip install -U torchao

In [7]:
non_instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [8]:
from datasets import load_dataset
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")

In [9]:
dataset[0]

{'Context': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?",
 'Response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is someh

In [10]:
def format_row(example):
    question = example["Context"]
    answer = example["Response"]
    example["Text"] = f"[INST] {question} [/INST] {answer}"
    return example

In [11]:
formatted_dataset = dataset.map(format_row)

In [12]:
formatted_dataset

Dataset({
    features: ['Context', 'Response', 'Text'],
    num_rows: 3512
})

In [13]:
import pandas as pd
df = pd.DataFrame(dataset)

In [14]:
df.to_csv("mental_health_counseling_conversations.csv", index=False)

In [15]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="mental_health_counseling_conversations.csv",split="train")
dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['Context', 'Response'],
    num_rows: 3512
})

In [17]:
def tokenize_fn(example):
    tokens = tokenizer(example["Text"], truncation=True, padding="max_length", max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [16]:
def format_example(example):
    prompt = f"### Instruction:\n{example['instruction']}\n### Input:\n{example['input']}\n### Response:\n{example['output']}"
    return {"text": prompt}

In [20]:
dataset = dataset.map(format_row)

Map:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [21]:
dataset

Dataset({
    features: ['Context', 'Response', 'Text'],
    num_rows: 3512
})

In [23]:
tokenized = dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [22]:
from peft import LoraConfig, get_peft_model, TaskType

In [24]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [25]:
model = get_peft_model(non_instruction_model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [26]:
args = TrainingArguments(
    output_dir="./tinyllama-instruction",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)


In [27]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
)

In [28]:
trainer.train()

Step,Training Loss
20,5.513592


KeyboardInterrupt: 